# Step 5: RQ1, which pollutants drive AQI, and how well can AQI be predicted?

Task: predict **today's AQI from today's pollutant readings** (no lagged AQI). Target is
log1p(AQI) because AQI is right-skewed; metrics are reported on the original AQI scale
(RMSE, MAE, R²), plus R² on the log scale.

Models: multiple linear regression, PCR, KNN, random forest, ANN. Tuned on validation (2019),
evaluated once on test (2020). Final models are trained on 2015-2018 only.

**Circularity, stated up front:** AQI is a fixed function of the pollutants (the maximum of the
CPCB sub-indices), so high R² is expected and is not the finding. The findings are (a) which
pollutants carry the signal and (b) which pollutant is the *binding* one by city and season (5.8).
Section 5.9 adds a non-circular version: tomorrow's AQI from today's data.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
try:                                   # Colab: get the repo (code + data) first
    import google.colab  # noqa: F401
    if not (ROOT / "src" / "aq.py").exists():
        import subprocess
        subprocess.run(["git", "clone", "-b", "claude/wizardly-meitner-nm2sfq",
                        "https://github.com/s-aura-bhhh/Air-quality-analysis.git"], check=True)
        ROOT = Path.cwd() / "Air-quality-analysis"
except ImportError:
    pass
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import aq

FIG, TAB = ROOT / "report" / "figures", ROOT / "report" / "tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

def save(name):
    plt.tight_layout(); plt.savefig(FIG / f"05_{name}.png", dpi=150); plt.show()

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
data, info = aq.build_dataset()
FEAT = info["rq1_features"]
NAMES = [c[2:] for c in FEAT]                      # drop the 'Z_' prefix for display
rq = data.dropna(subset=["y_log_aqi"]).sort_values("Date").reset_index(drop=True)
S = {p: rq[rq["period"] == p] for p in ["train", "val", "test"]}
S["test_nolock"] = S["test"][~S["test"]["lockdown"]]
X = {k: v[FEAT].values for k, v in S.items()}
y = {k: v["y_log_aqi"].values for k, v in S.items()}
print({k: len(v) for k, v in S.items()})

def metrics(y_log, pred_log):
    a, p = np.expm1(y_log), np.expm1(pred_log)
    return {"RMSE": np.sqrt(mean_squared_error(a, p)), "MAE": mean_absolute_error(a, p),
            "R2": r2_score(a, p), "R2_log": r2_score(y_log, pred_log)}

results = {}       # model name -> fitted predictor (function of X)

{'train': 13130, 'val': 5921, 'test': 2980, 'test_nolock': 1489}


## 5.1 Multiple linear regression (statsmodels)

In [3]:
ols = sm.OLS(y["train"], sm.add_constant(pd.DataFrame(X["train"], columns=NAMES))).fit()
coef = pd.DataFrame({"coef": ols.params, "ci_low": ols.conf_int()[0], "ci_high": ols.conf_int()[1],
                     "p_value": ols.pvalues}).drop("const")
coef = coef.reindex(coef["coef"].abs().sort_values(ascending=False).index).round(4)
coef.to_csv(TAB / "05_mlr_coefficients.csv")
print(f"train R2 (log scale) = {ols.rsquared:.3f}, adj R2 = {ols.rsquared_adj:.3f}, n = {int(ols.nobs)}")
coef

train R2 (log scale) = 0.815, adj R2 = 0.815, n = 13130


,coef,ci_low,ci_high,p_value
PM2.5,0.3634,0.3559,0.3710,0.0000
CO,0.2686,0.2627,0.2745,0.0000
PM10,0.1551,0.1461,0.1641,0.0000
SO2,0.0405,0.0344,0.0466,0.0000
NO,0.0329,0.0253,0.0404,0.0000
O3,0.0319,0.0259,0.0379,0.0000
NOx,0.0221,0.0149,0.0294,0.0000
NO2,-0.0208,-0.0279,-0.0138,0.0000
NH3,-0.0180,-0.0247,-0.0113,0.0000
Benzene,-0.0120,-0.0201,-0.0039,0.0039


Features are standardised, so coefficients are comparable: a one-standard-deviation increase in
log PM2.5 changes log AQI by `coef`. With n ≈ 13,000, almost every p-value is tiny; that says the
effects are not zero, not that they are large. Compare sizes, not p-values.

In [4]:
Xtr = pd.DataFrame(X["train"], columns=NAMES)
vif = pd.Series([variance_inflation_factor(sm.add_constant(Xtr).values, i + 1) for i in range(len(NAMES))],
                index=NAMES, name="VIF").sort_values(ascending=False).round(2)
vif.to_csv(TAB / "05_vif.csv")
vif

Toluene    2.62
NO         2.36
Benzene    2.35
PM10       2.32
PM2.5      2.24
NO2        2.00
NOx        1.82
SO2        1.46
CO         1.38
O3         1.37
NH3        1.36
Name: VIF, dtype: float64

VIF above 5 (some use 10) indicates a coefficient whose variance is inflated by collinearity: its
sign and size are unreliable even though predictions are fine. Expected: PM2.5/PM10 and NO/NO2/NOx.

In [5]:
fitted, resid = ols.fittedvalues, ols.resid
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hexbin(fitted, resid, gridsize=60, bins="log", cmap="viridis", mincnt=1)
axes[0].axhline(0, color="red", lw=1); axes[0].set_xlabel("fitted log AQI"); axes[0].set_ylabel("residual")
axes[0].set_title("Residuals vs fitted (train)")
sm.qqplot(resid, line="45", fit=True, ax=axes[1], markersize=2, alpha=0.3); axes[1].set_title("QQ plot of residuals")
save("mlr_residuals")
lr = LinearRegression().fit(X["train"], y["train"])
results["MLR"] = lr.predict

## 5.2 Principal component regression (PCR)

PCA on the standardised pollutants (train), regress log AQI on the first k components. k is chosen
by time-ordered cross-validation on train (`TimeSeriesSplit`, 5 folds: each fold trains on the
past and validates on the next block), then checked on validation.

In [6]:
pca_full = PCA().fit(X["train"])
ks = range(1, len(FEAT) + 1)
tscv = TimeSeriesSplit(n_splits=5)
cv_rmse = []
for k in ks:
    errs = []
    for a, b in tscv.split(X["train"]):
        m = make_pipeline(PCA(k), LinearRegression()).fit(X["train"][a], y["train"][a])
        errs.append(np.sqrt(mean_squared_error(y["train"][b], m.predict(X["train"][b]))))
    cv_rmse.append(np.mean(errs))
val_rmse = [np.sqrt(mean_squared_error(y["val"], make_pipeline(PCA(k), LinearRegression())
                                      .fit(X["train"], y["train"]).predict(X["val"]))) for k in ks]
k_best = int(np.argmin(cv_rmse)) + 1
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(ks, pca_full.explained_variance_ratio_, label="per component")
axes[0].plot(ks, np.cumsum(pca_full.explained_variance_ratio_), marker="o", color="C1", label="cumulative")
axes[0].set_xlabel("component"); axes[0].set_title("Scree plot (train pollutants)"); axes[0].legend()
axes[1].plot(ks, cv_rmse, marker="o", label="time-series CV (train)")
axes[1].plot(ks, val_rmse, marker="s", label="validation 2019")
axes[1].axvline(k_best, color="grey", ls="--"); axes[1].set_xlabel("k components")
axes[1].set_ylabel("RMSE (log AQI)"); axes[1].set_title("PCR: error vs number of components"); axes[1].legend()
save("pcr")
print("k chosen by CV:", k_best)
pcr = make_pipeline(PCA(k_best), LinearRegression()).fit(X["train"], y["train"])
results["PCR"] = pcr.predict

k chosen by CV: 10


**For the report:** if CV picks k = 11 (all components), PCR equals MLR: removing collinearity did
not improve prediction. That is the usual outcome with n ≫ p: collinearity hurts *interpretation*
of individual coefficients (VIF), not predictive accuracy. Say which happened here.

## 5.3 KNN, 5.4 random forest, 5.5 ANN: tuned on validation

In [7]:
def tune(make, grid, name):
    rows = []
    for params in grid:
        m = make(**params).fit(X["train"], y["train"])
        rows.append({**params, **metrics(y["val"], m.predict(X["val"])), "_model": m})
    t = pd.DataFrame(rows).sort_values("RMSE")
    best = t.iloc[0]
    print(f"{name}: best {dict((k, best[k]) for k in grid[0])}, val RMSE {best['RMSE']:.1f}")
    results[name] = best["_model"].predict
    return t.drop(columns="_model"), best["_model"]

knn_tab, knn = tune(lambda k: KNeighborsRegressor(k, n_jobs=-1),
                    [{"k": k} for k in [3, 5, 10, 20, 40, 80]], "KNN")
rf_tab, rf = tune(lambda n_estimators, max_depth: RandomForestRegressor(
                      n_estimators=n_estimators, max_depth=max_depth, min_samples_leaf=2,
                      n_jobs=-1, random_state=aq.SEED),
                  [{"n_estimators": n, "max_depth": d} for n in [100, 300] for d in [6, 12, None]], "RF")
ann_tab, ann = tune(lambda hidden, lr: MLPRegressor(hidden_layer_sizes=hidden, learning_rate_init=lr,
                        alpha=1e-3, early_stopping=True, max_iter=500, random_state=aq.SEED),
                    [{"hidden": h, "lr": l} for h in [(16,), (64,), (64, 32)] for l in [1e-3, 1e-2]], "ANN")
for t, nm in [(knn_tab, "knn"), (rf_tab, "rf"), (ann_tab, "ann")]:
    t.round(3).to_csv(TAB / f"05_tuning_{nm}.csv", index=False)

KNN: best {'k': np.int64(10)}, val RMSE 54.3


RF: best {'n_estimators': np.int64(100), 'max_depth': np.float64(12.0)}, val RMSE 49.5


ANN: best {'hidden': (64,), 'lr': np.float64(0.001)}, val RMSE 51.4


## 5.6 Comparison: all five models

In [8]:
rows = []
for name, pred in results.items():
    for part in ["val", "test", "test_nolock"]:
        rows.append({"model": name, "set": part, **metrics(y[part], pred(X[part]))})
comp = pd.DataFrame(rows).set_index(["model", "set"]).round(3).unstack("set")
comp = comp.reindex(columns=["val", "test", "test_nolock"], level=1)
comp.to_csv(TAB / "05_model_comparison.csv")
comp[["RMSE", "MAE", "R2"]]

RMSE                         MAE                         R2                   
set       val    test test_nolock     val    test test_nolock    val   test test_nolock
model                                                                                  
ANN    51.421  33.040      37.809  27.426  19.877      23.316  0.871  0.866       0.886
KNN    54.314  36.245      43.154  30.869  22.042      27.082  0.856  0.839       0.851
MLR    59.213  35.740      41.902  30.860  22.381      26.422  0.829  0.844       0.860
PCR    58.755  35.334      41.342  30.818  22.305      26.306  0.831  0.847       0.863
RF     49.542  29.496      33.642  23.257  17.078      19.725  0.880  0.894       0.909

In [9]:
fig, axes = plt.subplots(1, len(results), figsize=(4 * len(results), 4), sharex=True, sharey=True)
a = np.expm1(y["test"])
for ax, (name, pred) in zip(axes, results.items()):
    p = np.expm1(pred(X["test"]))
    ax.scatter(a, p, s=3, alpha=0.3); lim = np.percentile(a, 99.5)
    ax.plot([0, lim], [0, lim], color="red", lw=1); ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_title(name); ax.set_xlabel("actual AQI")
axes[0].set_ylabel("predicted AQI"); fig.suptitle("Test 2020: predicted vs actual")
save("pred_vs_actual")

**For the report:** do the non-linear models (RF, ANN, KNN) beat MLR/PCR by enough to matter?
AQI is a *maximum* of piecewise-linear sub-indices, a non-linear function, so tree models are
expected to fit it better. Compare test RMSE against the typical AQI level (~100-200) to judge if
the difference matters in practice.

## 5.7 What drives AQI: linear coefficients vs random forest importance

In [10]:
perm = permutation_importance(rf, X["val"], y["val"], n_repeats=5, random_state=aq.SEED, n_jobs=-1)
imp = pd.DataFrame({"MLR_std_coef": lr.coef_, "RF_impurity": rf.feature_importances_,
                    "RF_permutation_val": perm.importances_mean}, index=NAMES)
imp = imp.sort_values("RF_permutation_val", ascending=False).round(4)
imp.to_csv(TAB / "05_importance.csv")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
imp["MLR_std_coef"].plot(kind="barh", ax=axes[0], color="#4C72B0"); axes[0].invert_yaxis()
axes[0].set_title("MLR standardised coefficient"); axes[0].axvline(0, color="grey", lw=0.8)
imp["RF_permutation_val"].plot(kind="barh", ax=axes[1], color="#DD8452"); axes[1].invert_yaxis()
axes[1].set_title("RF permutation importance (validation)")
save("importance")
imp

,MLR_std_coef,RF_impurity,RF_permutation_val
PM2.5,0.3634,0.6998,0.6597
CO,0.2686,0.1719,0.3568
PM10,0.1551,0.0436,0.1286
O3,0.0319,0.0145,0.0370
NOx,0.0221,0.0264,0.0272
NO,0.0329,0.0083,0.0042
Toluene,-0.0045,0.0083,0.0037
NH3,-0.0180,0.0076,0.0023
NO2,-0.0208,0.0057,0.0001
SO2,0.0405,0.0073,-0.0008


Permutation importance (drop in validation fit when a feature is shuffled) is preferred over the
impurity importance, which is biased towards features with many split points. With correlated
features both methods split credit between them (PM2.5 vs PM10), so read the groups, not single
ranks.

## 5.8 The binding pollutant: which pollutant sets the AQI, by city and season

Using CPCB breakpoints, compute each pollutant's sub-index from the raw (non-imputed) daily
values; the pollutant with the highest sub-index is the one that sets the AQI that day.
Approximation: the official AQI uses 24-h rolling means and 8-h maxima for CO and O3 from hourly
data, so first check how well this daily recomputation matches the dataset's AQI.

In [11]:
raw = aq.load_checked()
raw = raw[raw["City"].isin(info["cities"])].copy()
si = aq.sub_indices(raw)
raw["AQI_recomputed"] = si.max(axis=1)
has_any = si.notna().any(axis=1)
raw["binding"] = si[has_any].idxmax(axis=1)
ok = raw["AQI"].notna() & raw["AQI_recomputed"].notna()
agree = (aq.aqi_to_bucket(raw.loc[ok, "AQI_recomputed"]) == raw.loc[ok, "AQI_Bucket"]).mean()
print(f"recomputed vs dataset AQI: r = {raw.loc[ok, ['AQI', 'AQI_recomputed']].corr().iloc[0, 1]:.3f}, "
      f"same bucket on {agree:.1%} of days")
season = raw["Date"].dt.month.map({12: "Winter (Dec-Feb)", 1: "Winter (Dec-Feb)", 2: "Winter (Dec-Feb)",
                                   3: "Summer (Mar-May)", 4: "Summer (Mar-May)", 5: "Summer (Mar-May)",
                                   6: "Monsoon (Jun-Sep)", 7: "Monsoon (Jun-Sep)", 8: "Monsoon (Jun-Sep)",
                                   9: "Monsoon (Jun-Sep)", 10: "Post-monsoon (Oct-Nov)",
                                   11: "Post-monsoon (Oct-Nov)"})
b = raw[ok].assign(season=season[ok])
by_city = pd.crosstab(b["City"], b["binding"], normalize="index").round(3)
by_city.to_csv(TAB / "05_binding_by_city.csv")
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(by_city * 100, annot=True, fmt=".0f", cmap="Blues", ax=ax, cbar_kws={"label": "% of days"})
ax.set_title("Binding pollutant (highest sub-index), % of days per city")
save("binding_by_city")
by_season = pd.crosstab(b["season"], b["binding"], normalize="index").round(3)
by_season.to_csv(TAB / "05_binding_by_season.csv")
(by_season * 100).round(1)

recomputed vs dataset AQI: r = 0.880, same bucket on 69.7% of days


binding,CO,NH3,NO2,O3,PM10,PM2.5,SO2
season,,,,,,,
Monsoon (Jun-Sep),24.9,0.0,2.6,4.0,41.8,26.3,0.4
Post-monsoon (Oct-Nov),13.8,0.2,1.2,1.1,26.9,56.7,0.1
Summer (Mar-May),11.3,0.0,0.4,4.3,47.5,36.1,0.2
Winter (Dec-Feb),7.5,0.1,0.3,0.6,28.9,62.6,0.0


In [12]:
# Unhealthy days only: what makes a day Poor or worse?
bad = b[b["AQI"] > aq.UNHEALTHY_AQI]
bad_tab = pd.crosstab(bad["City"], bad["binding"], normalize="index").mul(100).round(0)
bad_tab["n_unhealthy_days"] = bad.groupby("City").size()
bad_tab.to_csv(TAB / "05_binding_unhealthy_days.csv")
bad_tab

binding,CO,NO2,O3,PM10,PM2.5,n_unhealthy_days
City,,,,,,
Ahmedabad,89.0,0.0,0.0,0.0,10.0,1091
Amaravati,0.0,0.0,0.0,0.0,100.0,55
Amritsar,0.0,0.0,0.0,15.0,85.0,120
Bengaluru,73.0,0.0,0.0,10.0,17.0,41
Brajrajnagar,0.0,0.0,0.0,3.0,97.0,155
Chennai,20.0,0.0,0.0,0.0,80.0,127
Delhi,3.0,0.0,0.0,26.0,71.0,1301
Gurugram,0.0,0.0,0.0,5.0,94.0,753
Hyderabad,13.0,0.0,0.0,1.0,86.0,71


**For the report:** this is the main RQ1 finding. Name the pollutant that binds on most days
overall and on unhealthy days, the cities where it differs (Ahmedabad: CO), and the seasonal
shift (PM2.5 in winter and post-monsoon; PM10 or O3 in summer).

## 5.9 Non-circular version: tomorrow's AQI from today's data

Features: the RQ2 feature set (today's pollutants and AQI, lags, rolling mean, calendar, city).
Baseline: persistence, tomorrow's AQI = today's AQI.

In [13]:
F2 = info["rq2_features"]
nx = data.dropna(subset=["y_log_aqi_next"]).sort_values("Date")
P = {p: nx[nx["period"] == p] for p in ["train", "val", "test"]}
P["test_nolock"] = P["test"][~P["test"]["lockdown"]]
persist = lambda part: P[part]["aqi_z_filled"] * info["aqi_log_std"] + info["aqi_log_mean"]
lr2 = LinearRegression().fit(P["train"][F2], P["train"]["y_log_aqi_next"])
rf2 = RandomForestRegressor(n_estimators=300, max_depth=12, min_samples_leaf=2, n_jobs=-1,
                            random_state=aq.SEED).fit(P["train"][F2], P["train"]["y_log_aqi_next"])
rows = []
for part in ["val", "test", "test_nolock"]:
    yt = P[part]["y_log_aqi_next"].values
    rows += [{"model": "Persistence", "set": part, **metrics(yt, persist(part).values)},
             {"model": "MLR", "set": part, **metrics(yt, lr2.predict(P[part][F2]))},
             {"model": "RF", "set": part, **metrics(yt, rf2.predict(P[part][F2]))}]
nxt = pd.DataFrame(rows).set_index(["model", "set"]).round(3).unstack("set")
nxt = nxt.reindex(columns=["val", "test", "test_nolock"], level=1)
nxt.to_csv(TAB / "05_tomorrow_aqi.csv")
nxt[["RMSE", "MAE", "R2"]]

RMSE                         MAE                         R2                   
set             val    test test_nolock     val    test test_nolock    val   test test_nolock
model                                                                                        
MLR          63.031  41.514      52.007  28.129  20.392      25.398  0.803  0.786       0.782
Persistence  73.421  47.703      61.591  32.632  22.478      29.408  0.732  0.717       0.694
RF           55.072  38.092      46.798  26.206  18.958      23.259  0.849  0.820       0.823

**For the report:** this is the honest forecasting score. If the models only slightly beat
persistence, say so: most of tomorrow's AQI is today's AQI. Weather data (wind, rain, mixing
height) is what would improve this, and the dataset does not have it.

## 5.10 Sensitivity: results without Ahmedabad

In [14]:
d_na, info_na = aq.build_dataset(exclude_cities=["Ahmedabad"])
r_na = d_na.dropna(subset=["y_log_aqi"])
T = {p: r_na[r_na["period"] == p] for p in ["train", "val", "test"]}
lr_na = LinearRegression().fit(T["train"][FEAT], T["train"]["y_log_aqi"])
rf_na = RandomForestRegressor(**{k: rf.get_params()[k] for k in ["n_estimators", "max_depth", "min_samples_leaf"]},
                              n_jobs=-1, random_state=aq.SEED).fit(T["train"][FEAT], T["train"]["y_log_aqi"])
sens = pd.DataFrame({
    "MLR coef, all cities": lr.coef_, "MLR coef, no Ahmedabad": lr_na.coef_}, index=NAMES).round(3)
sens = sens.reindex(sens.iloc[:, 0].abs().sort_values(ascending=False).index)
sens.to_csv(TAB / "05_sensitivity_coef.csv")
print("test, no Ahmedabad:  MLR", {k: round(v, 3) for k, v in metrics(T["test"]["y_log_aqi"], lr_na.predict(T["test"][FEAT])).items()})
print("test, no Ahmedabad:  RF ", {k: round(v, 3) for k, v in metrics(T["test"]["y_log_aqi"], rf_na.predict(T["test"][FEAT])).items()})
sens

test, no Ahmedabad:  MLR {'RMSE': np.float64(28.266), 'MAE': 20.065, 'R2': 0.827, 'R2_log': 0.803}
test, no Ahmedabad:  RF  {'RMSE': np.float64(22.98), 'MAE': 15.419, 'R2': 0.886, 'R2_log': 0.864}


,"MLR coef, all cities","MLR coef, no Ahmedabad"
PM2.5,0.363,0.395
CO,0.269,0.135
PM10,0.155,0.160
SO2,0.041,0.018
NO,0.033,0.036
O3,0.032,0.042
NOx,0.022,0.020
NO2,-0.021,-0.037
NH3,-0.018,-0.015
Benzene,-0.012,-0.011


**For the report:** if CO's coefficient drops sharply without Ahmedabad, then the "CO drives AQI"
signal in the full model is mostly one city's (suspect) sensor. Report both.